# Генератор датасета для классификации команд

Датасет в формате JSONL должен содержать обучающие данные для классификационной модели (например, BERT) в следующем формате:
```json
{
    "text": "Сделай громкость потише",
    "command": {
        "opcode": "SETTINGS_VOLUME",
        "recognizedArgs": {
            "group": "master",
            "action": "decrease"
        }
    }    
}
```

## Допустимые команды
Условные обозначения:
- Код операции (`opcode`): тип команды
- Распознаваемые аргументы (`recognizedArgs`): аргументы, распознанные с помощью классификационной модели для данного типа команды
- Контекстные аргументы (`contextArgs`): аргументы, передаваемые в веб-запросе и не зависящие от типа команды, но используемые для её исполнения

| Код операции      | Распознаваемые аргументы                                                                               | Контекстные аргументы | Описание                                                               |
|-------------------|--------------------------------------------------------------------------------------------------------|-----------------------|------------------------------------------------------------------------|
| `HINT_NEAREST`    | Нет                                                                                                    | TBA                   | Активировать подсказку к ближайшему заданию (если доступна)            |
| `SETTINGS_VOLUME` | `group: Literal["master", "music", "sfx", "voice"]`, `action: Literal["increase", "decrease", "mute"]` | TBA                   | Увеличить / уменьшить / заглушить громкость для выбранной группы аудио |
| `PROGRESS_LEVEL`  | Нет                                                                                                    | TBA                   | Вывести информацию о текущем уровне игрока и целях на нём              |
| `FACT_RANDOM`     | `target: Literal["logic", "lore"]`                                                                     | TBA                   | Вывести случайный факт о троичной логике или вселенной игры            |
| `UNKNOWN`         | Нет                                                                                                    | Нет                   | Возвращается, если не удалось распознать команду                       | 

In [14]:
import random
import json
import pymorphy2
import re
from typing import Any, Tuple, List, Dict, Set, Literal, Union, Optional

from app.services.command.classifier.command_models import Command, CommandOpcode, SettingsVolumeRecognizedArgs, FactRandomRecognizedArgs

In [15]:
DATASET_PATH = "./data/command_dataset.jsonl"

In [16]:
rng = random.Random(x=267)
WORD_REGEX = re.compile(r'\b\w+(?:-\w+)*\b')
COUNT = 8192
KNOWN_COUNT = int(COUNT * 0.8)

In [17]:
morph = pymorphy2.MorphAnalyzer()

def inflect_noun_or_adj(
    word: str,
    case: Literal['nomn', 'gent', 'gen1', 'gen2', 'datv', 'accs', 'ablt', 'loct', 'loc1', 'loc2'] = 'nomn',
    gender: Literal['masc', 'femn', 'neut', 'ms-f'] = 'masc',
    number: Literal['sing', 'plur'] = 'sing'        
) -> str:
    parsed_list = morph.parse(word)
    
    if not parsed_list:
        return word
    
    # Pick the first match (usually the most probable)
    parsed = parsed_list[0]
    
    # For adjectives/participles, gender is ignored if number is 'plur'
    target_grammemes = {case, number}
    if number == 'sing':
        target_grammemes.add(gender)
        
    inflected = parsed.inflect(target_grammemes)
    
    return inflected.word if inflected else word

print(inflect_noun_or_adj('троичная', case='ablt', number='plur'))
print(inflect_noun_or_adj('импликация', case='ablt', number='plur'))
print(inflect_noun_or_adj('задание', case='datv', gender='neut'))

троичными
импликациями
заданию


In [18]:
def generate_command_definition(
        include_unknown: bool = True,
        rng: Optional[random.Random] = None
) -> Tuple[CommandOpcode, Optional[Dict[str, Any]]]:
    _rng = rng if rng is not None else random
    
    AVAILABLE_COMMANDS = [
        (CommandOpcode.HINT_NEAREST, None),
        (CommandOpcode.SETTINGS_VOLUME, {
            'group': _rng.choice(['master', 'music', 'sfx', 'voice']),
            'action': _rng.choice(['increase', 'decrease', 'mute']),
        }),
        (CommandOpcode.PROGRESS_LEVEL, None),
        (CommandOpcode.FACT_RANDOM, {
            'target': _rng.choice(['logic', 'lore'])
        }),
    ]
    
    if include_unknown:
        AVAILABLE_COMMANDS.append((CommandOpcode.UNKNOWN, None))
    
    return _rng.choice(AVAILABLE_COMMANDS)

for i in range(16):
    command = generate_command_definition(rng)
    print(f'{command[0].name}: {command[1]}')
    # Command.validate({'opcode': command[0], 'recognizedArgs': command[1], 'contextArgs': None})

SETTINGS_VOLUME: {'group': 'voice', 'action': 'decrease'}
PROGRESS_LEVEL: None
SETTINGS_VOLUME: {'group': 'master', 'action': 'mute'}
FACT_RANDOM: {'target': 'logic'}
UNKNOWN: None
SETTINGS_VOLUME: {'group': 'voice', 'action': 'increase'}
UNKNOWN: None
SETTINGS_VOLUME: {'group': 'master', 'action': 'increase'}
UNKNOWN: None
HINT_NEAREST: None
UNKNOWN: None
UNKNOWN: None
SETTINGS_VOLUME: {'group': 'voice', 'action': 'decrease'}
PROGRESS_LEVEL: None
FACT_RANDOM: {'target': 'lore'}
PROGRESS_LEVEL: None


In [19]:
FILLERS_START: List[Tuple[str, int]] = [  # With percentages for weighted choices
    ('', 40),  # 40%
    ('пожалуйста', 10), ('эй', 5),  ('слушай', 5),  # 20%
    ('помощник', 8), ('ассистент', 8), ('шаттл', 3), ('корабль', 1),  # 20%
    ('компьютер', 6), ('товарищ компьютер', 2), ('бортовой компьютер', 2),  # 10%
    ('друг', 5), ('товарищ', 3), ('камрад', 2),  # 10%
]
FILLERS_END: List[Tuple[str, int]] = [  # With percentages for weighted choices
    ('', 70),  # 70%
    ('пожалуйста!', 8), ('плиз!', 4), ('будь добр.', 3),  # 15%
    ('быстро!', 8), ('немедленно.', 4), ('сейчас же!', 3)  # 15%
]

MESSAGE_TEMPLATES_FOR_OPCODE = {
    CommandOpcode.HINT_NEAREST: [
        'мне нужна подсказка {nearest_datv}',
        'нужна помощь {nearest_ablt}',
        'дай мне подсказку {nearest_datv}',
        'подсказка {nearest_datv}',
        'помоги мне {nearest_ablt}'
    ],
    CommandOpcode.SETTINGS_VOLUME: [
        'нужно {modify_verb_infv} {audio_group_accs}',
        'нужно {modify_verb_infv} громкость {audio_group_gent}',
        '{modify_verb_impr} {audio_group_accs}',
        '{audio_group_nomn} {must} быть {modify_verb_prts}',
        'сделай {audio_group_accs} {modify_adverb}',
        '{modify_verb_impr} громкость {audio_group_gent}',
        'громкость {audio_group_gent} надо {modify_verb_infv}'
    ],
    CommandOpcode.PROGRESS_LEVEL: [
        'где я',
        '{locate_verb_impr} где я',
        '{locate_verb_impr} {my_pron_accs} {location_noun_accs}',
        'нужно {locate_verb_infv} {my_pron_accs} {location_noun_accs}',
        'в {which_adj_loct} я {level_noun_loct}',
        '{which_adj_short_nomn} {my_pron_nomn} {level_noun_nomn}',
        '{which_adj_short_nomn} {my_pron_nomn} {location_noun_nomn}',
        'куда я попал'
    ],
    CommandOpcode.FACT_RANDOM: [
        '{tell_verb_impr} мне {fact_noun_accs} про {target_full_accs}',
        '{tell_verb_impr} {fact_noun_accs} о {target_full_loct}',
        '{want_verb_face1} {know_verb_infv} про {target_full_accs}',
        '{fact_noun_accs} о {target_full_loct}',
        '{needed} {fact_noun_nomn} по теме - {target_full_nomn}'
    ],
    CommandOpcode.UNKNOWN: [
        'обстановка по кайфу',
        'как дела',
        'как себя чувствуешь',
        'как сам',
        'всё у меня отлично'
    ]
}

MESSAGE_TEMPLATES_MISC = {
    'to_nearest_task_datv': ['', 'к {to_nearest_noun}', 'к {to_nearest_adj} {to_nearest_noun}', 'по {to_nearest_noun}'],
    'to_nearest_task_ablt': ['с {to_nearest_noun}', 'с {to_nearest_adj} {to_nearest_noun}']
}

MISC_WORDS = {
    'task': {
        'femn': ['задача', 'задачка', 'головоломка', 'цепь'],
        'neut': ['задание', 'уравнение'],
    },
    'nearest': ['ближайший', 'близкий', 'текущий', 'этот'],
    'audio_decrease_verb': ['уменьшить', 'снизить', 'понизить', 'ослабить'],
    'audio_increase_verb': ['увеличить', 'прибавить', 'повысить', 'усилить'],
    'audio_mute_verb': ['глушить', 'заглушить', 'выключить', 'отключить'],
    'audio_decrease_adverb': ['потише', 'тише', 'тихо', 'послабее'],
    'audio_increase_adverb': ['погромче', 'громче', 'громко', 'посильнее'],
    'audio_mute_adverb': ['беззвучно', 'в ноль'],
    'audio_master_group_nomn_accs': {
        'neut': ['аудио'], 'masc': ['звук', 'весь звук', 'общий звук']
    },
    'audio_music_group_noun': ['музыка', 'музычка', 'мелодия'],
    'audio_sfx_group_both': ['звуковые эффекты', 'шумовые эффекты', 'аудиоэффекты'],
    'audio_voice_group_noun': {
        'masc': ['голос', 'разговор'],
        'femn': ['речь'],
        'plur': ['речи', 'голоса', 'разговоры']
    },
    'audio_master_group_gent': ['общего звука', 'всего звука'],
    'audio_music_group_gent': ['музыки', 'мелодий'],
    'audio_sfx_group_gent': ['аудиоэффектов', 'звуковых эффектов'],
    'audio_voice_group_gent': ['голосов', 'речи', 'разговоров'],
    'must': {
        'masc': 'должен', 'femn': 'должна', 'neut': 'должно', 'plur': 'должны'
    },
    'needed': {
        'masc': 'нужен', 'femn': 'нужна', 'neut': 'нужно', 'plur': 'нужны'
    },
    'locate_verb': ['найти', 'определить', 'указать', 'выяснить'],
    'location_noun': {
        'neut': ['расположение', 'местоположение', 'местонахождение'],
        'femn': ['локация'],
        'masc': ['уровень']
    },
    'level_noun': {
        'masc': ['уровень', 'отсек', 'сектор', 'зал'],
        'femn': ['комната'],
        'neut': ['отделение']
    },
    'which_adj': {
        'loct': {
            'masc': 'каком', 'femn': 'какой', 'neut': 'каком', 'plur': 'каких'
        },
    },
    'which_adj_short_nomn': {
        'masc': 'каков', 'femn': 'какова', 'neut': 'каково', 'plur': 'каковы'
    },
    'my_pron': {
        'nomn': { 'masc': 'мой', 'femn': 'моя', 'neut': 'моё', 'plur': 'мои' },
        'accs': { 'masc': 'мой', 'femn': 'мою', 'neut': 'моё', 'plur': 'мои' },
    },
    'information_noun': {
        'masc': ['факт'],
        'femn': ['информация'],
        'neut': ['знание'],
        'plur': ['сведения', 'знания', 'факты'],
    },
    'target_logic_full': {
        'nomn': [
            'троичная логика', 'логика', 'арифметика', 'нонарная арифметика', 'девятеричная арифметика', 'тернарная логика', 'троичные вычисления', 'троичные схемы', 'схемотехника', 'нонарная система',
        ],
        'accs': [
            'троичную логику', 'логику', 'арифметику', 'нонарную арифметику', 'девятеричную арифметику', 'тернарную логику', 'троичные вычисления', 'троичные схемы', 'схемотехнику', 'нонарную систему',
        ],
        'loct': [
            'троичной логике', 'логике', 'арифметике', 'нонарной арифметике', 'девятеричной арифметике', 'тернарной логике', 'троичных вычислениях', 'троичных схемах', 'схемотехнике', 'нонарной системе',
        ],
    },
    'target_lore_full': {
        'nomn': [
            'события', 'случившееся', 'события на станции', 'состояние станции', 'произошедшее', 'история', 'хроники', 'предыстория', 'космическая программа', 'Пульсар', 'Гиперион', 'экспедиция'
        ],
        'accs': [
            'подоплёку событий', 'историю', 'хроники', 'предысторию', 'космическую программу', 'Пульсара', 'Гипериона'
        ],
        'loct': [
            'событиях', 'случившемся', 'событиях станции', 'состоянии станции', 'произошедшем', 'истории', 'хрониках', 'предыстории', 'Пульсаре', 'Гиперионе', 'экспедиции'
        ],
    },
    'want_verb_face1': ['хочу', 'желаю', 'хотел бы'],
    'know_verb_infv': ['знать', 'знать всё', 'знать больше', 'понимать всё', 'узнать'],
    'tell_verb_impr': ['расскажи', 'скажи', 'объясни'],
}

In [20]:
def mutate_text(text: str, rng: Optional[random.Random] = None) -> str:
    _rng = rng if rng is not None else random
    
    # substitute letters
    letter_replacement = {
        'ы': ['и', 'у', 'э'], 'у': ['о', 'а', 'ы'], 'а': ['о', 'э'], 'о': ['а', 'у'], 'ъ': ['ь'], 'ь': ['ъ'], ' ': ['', ' ']
    }
    text = ''.join([
        _rng.choice(letter_replacement.get(c, [c])) if _rng.random() < 0.05 else c for c in text 
    ])
    
    # insert words between words
    stopwords = ['ну', 'эм', 'эмм', 'как бы', 'мда', 'значит', 'типа']
    words = [s for s in re.split('\s+', text) if s]
    if _rng.random() < 0.25:
        words.insert(_rng.randint(0, len(words) - 1), _rng.choice(stopwords))
    text = ' '.join(words)
    
    # insert up to 5 random spaces
    length = len(text)
    chars = [c for c in text]
    for i in range(_rng.randint(0, 5)):
        if _rng.random() < 0.05:
            pos = _rng.randint(0, length)
            chars.insert(pos, ' ')
            
    text = ''.join(chars).strip()
    return text


for i in range(10):
    text = "Пожалуйста, расскажи мне факты про троичные вычисления, друг."
    print(mutate_text(text, rng))

Пожалыйсто, рэсскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про троичные вычисления, значит друг.
Пожалуйста, расскажи мне факты про троичные вычисления, дрог.
Пожалуйста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты как бы про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про троичные вычисления, друг.
Пожалуйста, расскажи мне факты про ну троичные вычисления, друг.
Пожалуйста, расскажи мне факты про труичные вычисления, друг.


In [21]:
def generate_command_text_body(
    opcode: CommandOpcode,
    recognized_args: Optional[Dict[str, Any]] = None,
    rng: Optional[random.Random] = None,     
) -> str:
    _rng = rng if rng is not None else random
    
    match opcode:
        case CommandOpcode.HINT_NEAREST:
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.HINT_NEAREST])
            
            to_nearest_case = _rng.choice(['datv', 'ablt'])
            
            to_nearest_gender = _rng.choice(list(MISC_WORDS['task'].keys()))
            to_nearest_adj_ablt = inflect_noun_or_adj(
                _rng.choice(MISC_WORDS['nearest']),
                case='ablt',
                gender=to_nearest_gender
            )
            to_nearest_noun_ablt = inflect_noun_or_adj(
                _rng.choice(MISC_WORDS['task'][to_nearest_gender]), 
                case='ablt',
                gender=to_nearest_gender
            )
            to_nearest_adj_datv = inflect_noun_or_adj(
                _rng.choice(MISC_WORDS['nearest']),
                case='datv',
                gender=to_nearest_gender
            )
            to_nearest_noun_datv = inflect_noun_or_adj(
                _rng.choice(MISC_WORDS['task'][to_nearest_gender]), 
                case='datv',
                gender=to_nearest_gender
            )
            nearest_datv = _rng.choice(MESSAGE_TEMPLATES_MISC[f'to_nearest_task_datv']).format(
                to_nearest_adj=to_nearest_adj_datv,
                to_nearest_noun=to_nearest_noun_datv
            )
            nearest_ablt = _rng.choice(MESSAGE_TEMPLATES_MISC[f'to_nearest_task_ablt']).format(
                to_nearest_adj=to_nearest_adj_ablt,
                to_nearest_noun=to_nearest_noun_ablt
            )
            
            text = template.format(
                nearest_datv=nearest_datv,
                nearest_ablt=nearest_ablt
            ).strip()
            return text
        
        case CommandOpcode.SETTINGS_VOLUME:
            args = SettingsVolumeRecognizedArgs.model_validate(recognized_args)
            
            match args.group:
                case 'master':
                    gender = _rng.choice(list(MISC_WORDS['audio_master_group_nomn_accs'].keys()))
                case 'music':
                    gender = "femn"
                case 'sfx':
                    gender = "plur"
                case 'voice':
                    gender = _rng.choice(list(MISC_WORDS['audio_voice_group_noun'].keys()) + ['plur'])
                case _:
                    gender = 'plur'
            
            verb = _rng.choice(MISC_WORDS[f"audio_{args.action}_verb"])
            parsed_verb_list = morph.parse(verb)
            if parsed_verb_list:
                parsed_verb = parsed_verb_list[0]
                verb_infv = parsed_verb.inflect({ 'INFN' }).word
                verb_impr = parsed_verb.inflect({ 'impr', 'excl', ('sing' if _rng.random() < 0.5 else 'plur') }).word
                verb_prts = parsed_verb.inflect({ 'PRTS', 'pssv', 'past', gender }).word
            else:
                verb_infv = verb
                verb_impr = verb
                verb_prts = verb
                
            adverb = _rng.choice(MISC_WORDS[f"audio_{args.action}_adverb"])
                
            must_word = MISC_WORDS['must'][gender]
            
            match args.group: 
                case 'master':
                    audio_group_nomn = _rng.choice(MISC_WORDS['audio_master_group_nomn_accs'][gender])
                    audio_group_acc = audio_group_nomn
                case 'music':
                    word = _rng.choice(MISC_WORDS['audio_music_group_noun'])
                    parsed_list = morph.parse(word)
                    parsed = parsed_list[0]
                    audio_group_nomn = word
                    audio_group_acc = parsed.inflect({ 'accs' }).word
                case 'voice':
                    word = _rng.choice(MISC_WORDS['audio_voice_group_noun'][gender])
                    parsed_list = morph.parse(word)
                    parsed = parsed_list[0]
                    audio_group_nomn = word
                    audio_group_acc = parsed.inflect({ 'accs' }).word
                case 'sfx':
                    words = _rng.choice(MISC_WORDS['audio_sfx_group_both'])
                    audio_group_nomn = words
                    audio_group_acc = words
                case _:
                    audio_group_nomn = '[...]'
                    audio_group_acc = '[...]'
            
            audio_group_gent = _rng.choice(MISC_WORDS[f'audio_{args.group}_group_gent'])
                
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.SETTINGS_VOLUME])
            text = template.format(
                modify_verb_infv=verb_infv,
                modify_verb_impr=verb_impr,
                modify_verb_prts=verb_prts,
                modify_adverb=adverb,
                audio_group_nomn=audio_group_nomn,
                audio_group_accs=audio_group_acc,
                audio_group_gent=audio_group_gent,
                must=must_word
            )
            return text
        
        case CommandOpcode.PROGRESS_LEVEL:
            parsed_verb = morph.parse(_rng.choice(MISC_WORDS['locate_verb']))[0]
            verb_infv = parsed_verb.inflect({ 'INFN' }).word
            verb_impr = parsed_verb.inflect({ 'impr', 'excl', ('sing' if _rng.random() < 0.5 else 'plur') }).word
            
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.PROGRESS_LEVEL])
            
            gender = (
                _rng.choice(list(MISC_WORDS['location_noun'].keys()))
                if 'location' in template
                else _rng.choice(list(MISC_WORDS['level_noun'].keys()))
            )
            
            my_pron_nomn = MISC_WORDS['my_pron']['nomn'][gender]
            my_pron_accs = MISC_WORDS['my_pron']['accs'][gender]
            
            parsed_location_noun = morph.parse(_rng.choice(MISC_WORDS['location_noun'][gender]))[0]
            parsed_level_noun = morph.parse(_rng.choice(MISC_WORDS['level_noun'][gender]))[0]
            location_noun_nomn = parsed_location_noun.inflect({ 'nomn', gender }).word
            location_noun_accs = parsed_location_noun.inflect({ 'accs', gender }).word
            level_noun_nomn = parsed_level_noun.inflect({ 'nomn', gender }).word
            level_noun_loct = parsed_level_noun.inflect({ 'loct', gender }).word
            which_adj_loct = MISC_WORDS['which_adj']['loct'][gender]
            which_adj_short_nomn = MISC_WORDS['which_adj_short_nomn'][gender]
            
            text = template.format(
                locate_verb_infv=verb_infv,
                locate_verb_impr=verb_impr,
                level_noun_nomn=level_noun_nomn,
                level_noun_loct=level_noun_loct,
                my_pron_nomn=my_pron_nomn,
                my_pron_accs=my_pron_accs,
                location_noun_nomn=location_noun_nomn,
                location_noun_accs=location_noun_accs,
                which_adj_loct=which_adj_loct,
                which_adj_short_nomn=which_adj_short_nomn
            )
            return text
        
        case CommandOpcode.FACT_RANDOM:
            # CommandOpcode.FACT_RANDOM: [
            #     '{tell_verb_impr} мне {fact_noun_accs} про {target_full_accs}',
            #     '{tell_verb_impr} {fact_noun_accs} о {target_full_loct}',
            #     '{want_verb_face1} {know_verb_infv} больше про {target_full_accs}',
            #     '{fact_noun_accs} о {target_full_loct}',
            #     '{needed} {fact_noun_nomn} по теме - {target_full_nomn}'
            # ],
            args = FactRandomRecognizedArgs.model_validate(recognized_args)
            
            target_full_nomn = _rng.choice(MISC_WORDS[f'target_{args.target}_full']['nomn'])
            target_full_accs = _rng.choice(MISC_WORDS[f'target_{args.target}_full']['accs'])
            target_full_loct = _rng.choice(MISC_WORDS[f'target_{args.target}_full']['loct'])
            
            template = _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.FACT_RANDOM])
            
            gender = _rng.choice(list(MISC_WORDS['information_noun'].keys()))
            needed_verb = MISC_WORDS['needed'][gender]
            
            fact_noun = _rng.choice(MISC_WORDS['information_noun'][gender])
            fact_parsed = morph.parse(fact_noun)[0]
            fact_noun_nomn = fact_noun
            fact_noun_accs = fact_parsed.inflect({ 'accs', gender }).word
            
            text = template.format(
                tell_verb_impr=_rng.choice(MISC_WORDS['tell_verb_impr']),
                fact_noun_nomn=fact_noun_nomn,
                fact_noun_accs=fact_noun_accs,
                know_verb_infv=_rng.choice(MISC_WORDS['know_verb_infv']),
                target_full_nomn=target_full_nomn,
                target_full_accs=target_full_accs,
                target_full_loct=target_full_loct,
                needed=needed_verb,
                want_verb_face1=_rng.choice(MISC_WORDS['want_verb_face1']),
            )
            return text
        
        case _:
            return _rng.choice(MESSAGE_TEMPLATES_FOR_OPCODE[CommandOpcode.UNKNOWN])

In [22]:
def generate_command_text(
    opcode: CommandOpcode,
    recognized_args: Optional[Dict[str, Any]] = None,
    body_text: Optional[str] = None,
    mutate: bool = False,
    rng: Optional[random.Random] = None,
) -> str:
    _rng = rng if rng is not None else random
    
    def unpair(list_pairs: List[Tuple[Any, Any]]) -> Tuple[List[Any], List[Any]]:
        return [x[0] for x in list_pairs], [x[1] for x in list_pairs]
    
    text = ""
    
    prefix_options, prefix_weights = unpair(FILLERS_START)
    suffix_options, suffix_weights = unpair(FILLERS_END)
    
    prefix = _rng.choices(prefix_options, prefix_weights, k=1)[0]
    suffix = _rng.choices(suffix_options, suffix_weights, k=1)[0]
    
    if prefix:
        prefix += ", "
    if suffix:
        suffix = ", " + suffix
      
    if body_text is None:
        body = generate_command_text_body(opcode, recognized_args, rng)
    else:
        body = body_text
    
    text = prefix + body + suffix
    
    if mutate:
        text = mutate_text(text, _rng)
    
    return text
    
for i in range(32):
    # opcode = rng.choice(list(CommandOpcode))
    opcode, args = generate_command_definition(include_unknown=False, rng=rng)
    body_text = generate_command_text_body(opcode, recognized_args=args, rng=rng)
    text = generate_command_text(opcode, recognized_args=args, body_text=body_text, rng=rng)
    print(f"{opcode} () -> {text}")

FACT_RANDOM () -> ассистент, объясни знание о троичных вычислениях
SETTINGS_VOLUME () -> ослабь разговоры
HINT_NEAREST () -> помоги мне с головоломкой, пожалуйста!
PROGRESS_LEVEL () -> эй, укажите где я
PROGRESS_LEVEL () -> бортовой компьютер, куда я попал
HINT_NEAREST () -> друг, дай мне подсказку
FACT_RANDOM () -> нужны знания по теме - нонарная арифметика
FACT_RANDOM () -> эй, расскажи мне факт про Гипериона
SETTINGS_VOLUME () -> слушай, выключите громкость музыки, пожалуйста!
HINT_NEAREST () -> дай мне подсказку по головоломке
PROGRESS_LEVEL () -> найдите где я
FACT_RANDOM () -> шаттл, сведения о арифметике
FACT_RANDOM () -> пожалуйста, объясни мне факты про схемотехнику
PROGRESS_LEVEL () -> где я
PROGRESS_LEVEL () -> нужно указать мой уровень, немедленно.
SETTINGS_VOLUME () -> пожалуйста, нужно глушить громкость голосов, плиз!
HINT_NEAREST () -> друг, помоги мне с этим уравнением
FACT_RANDOM () -> скажи знание о хрониках
FACT_RANDOM () -> помощник, факты о хрониках
SETTINGS_VOLUME

In [23]:
for i in range(32):
    # opcode = rng.choice(list(CommandOpcode))
    opcode, args = (CommandOpcode.FACT_RANDOM, {'target': 'logic'})
    text = generate_command_text(opcode, recognized_args=args, rng=rng)
    print(text)

эй, факт о нонарной арифметике, немедленно.
пожалуйста, скажи мне факт про схемотехнику
факт о тернарной логике
скажи факт о девятеричной арифметике
друг, нужна информация по теме - тернарная логика
пожалуйста, нужна информация по теме - нонарная система
пожалуйста, хочу знать про троичные вычисления
знание о троичных вычислениях, будь добр.
друг, факт о арифметике
нужен факт по теме - схемотехника
расскажи знание о девятеричной арифметике
ассистент, скажи знание о нонарной системе
компьютер, желаю узнать про троичные схемы
пожалуйста, объясни знание о нонарной арифметике
пожалуйста, нужен факт по теме - тернарная логика
хотел бы знать про схемотехнику
компьютер, скажи мне знание про тернарную логику
пожалуйста, объясни факт о троичных вычислениях, быстро!
знание о нонарной арифметике
объясни факты о тернарной логике, немедленно.
нужен факт по теме - девятеричная арифметика
пожалуйста, нужны факты по теме - тернарная логика
товарищ, расскажи мне знания про тернарную логику
ассистент, о

In [24]:
file = open(DATASET_PATH, "a+", encoding='utf8')

words = set()

objects = []

for i in range(KNOWN_COUNT):
    opcode, args = generate_command_definition(include_unknown=False, rng=rng)
    body_text = generate_command_text_body(opcode, recognized_args=args, rng=rng)
    text = generate_command_text(
        opcode,
        recognized_args=args,
        body_text=body_text,
        mutate=(rng.random() < 0.1),
        rng=rng
    )
    
    for word in WORD_REGEX.findall(text):
        if rng.random() < 0.5:
            words.add(word)
        else: 
            words.add(word[::-1])
    
    objects.append({ 'text': text, 'command': { 'opcode': opcode.name, 'recognizedArgs': args }})

words = list(words)

for i in range(COUNT - KNOWN_COUNT):
    garbage_text = generate_command_text_body(CommandOpcode.UNKNOWN, recognized_args=None, rng=rng)
    garbage_text = re.split('\s+', garbage_text)
    garbage_text_length = len(garbage_text)
    for i in range(rng.randint(0, 1)):
        if rng.random() < 0.05:
            garbage_text.insert(rng.randint(0, garbage_text_length - 1), rng.choice(words))
            garbage_text_length += 1
    garbage_text = ' '.join(garbage_text)        
    
    text = generate_command_text(
        CommandOpcode.UNKNOWN,
        recognized_args=None,
        body_text=garbage_text,
        mutate=(rng.random() < 0.35),
        rng=rng
    )
    objects.append({ 'text': text, 'command': { 'opcode': CommandOpcode.UNKNOWN, 'recognizedArgs': None }})

rng.shuffle(objects)

for obj in objects:
    json.dump(fp=file, obj=obj, ensure_ascii=False)
    file.write('\n')
    
file.close()